In [1]:
import os
import pandas as pd
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from gensim.models import Word2Vec

In [2]:
os.chdir("../")

In [3]:
from src.functions import remove_stopwords_punctuation, remove_outliers, lematiza_tokens, embedding_w2v

In [4]:
df = pd.read_csv("./data/buscape.csv")

In [5]:
df

,original_index,review_text,review_text_processed,review_text_tokenized,polarity,rating,kfold_polarity,kfold_rating
0,4_55516,"Estou muito satisfeito, o visor é melhor do qu...","estou muito satisfeito, o visor e melhor do qu...","['estou', 'muito', 'satisfeito', 'visor', 'mel...",1.0,4,1,1
1,minus_1_105339,"""muito boa\n\nO que gostei: preco\n\nO que não...","""muito boa\n\no que gostei: preco\n\no que nao...","['muito', 'boa', 'que', 'gostei', 'preco', 'qu...",1.0,5,1,1
2,23_382139,"Rápida, ótima qualidade de impressão e fácil d...","rapida, otima qualidade de impressao e facil d...","['rapida', 'otima', 'qualidade', 'de', 'impres...",1.0,5,1,1
3,2_446456,Produto de ótima qualidade em todos os quesito!,produto de otima qualidade em todos os quesito!,"['produto', 'de', 'otima', 'qualidade', 'em', ...",1.0,5,1,1
4,0_11324,Precisava comprar uma tv compatível com meu dv...,precisava comprar uma tv compativel com meu dv...,"['precisava', 'comprar', 'uma', 'tv', 'compati...",1.0,5,1,1
...,...,...,...,...,...,...,...,...
84986,1_422965,"Produto muito bom, simples e barato","produto muito bom, simples e barato","['produto', 'muito', 'bom', 'simples', 'barato']",1.0,5,10,10
84987,minus_1_150466,O esquema antigo de desmontagem e limpeza das ...,o esquema antigo de desmontagem e limpeza das ...,"['esquema', 'antigo', 'de', 'desmontagem', 'li...",NaN,3,-1,10
84988,0_414799,Esse jogo é muito maneiro é um jogo onde vc te...,esse jogo e muito maneiro e um jogo onde vc te...,"['esse', 'jogo', 'muito', 'maneiro', 'um', 'jo...",1.0,5,10,10
84989,0_389898,Muito bom e intuitivo!\n\nO que gostei: Educa ...,muito bom e intuitivo!\n\no que gostei: educa ...,"['muito', 'bom', 'intuitivo', 'que', 'gostei',...",NaN,3,-1,10


### 1. Train-Test Split

In [6]:
X_data = df.drop(columns=['polarity'])
y_data = df['polarity']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, random_state=42)

## Conjunto de Treino

### 2. Limpeza de Dados
##### 2.1. Tratamento de Nulos

In [8]:
X_train.shape

(59493, 7)

In [9]:
X_train['review_text'].isnull().sum()

np.int64(1)

In [10]:
X_train = X_train.dropna(subset=['review_text'])

In [11]:
X_train.shape

(59492, 7)

In [12]:
# paridade de índices
y_train = y_train.loc[X_train.index]

In [13]:
y_train.shape

(59492,)

##### 2.2. Tratamento de Outliers

In [14]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [15]:
X_train_token = pd.DataFrame()
X_train_token['token'] = X_train['review_text']
X_train_token

,token
69382,Ótimo aparelho. Para quem já usa ou irá experi...
11241,"Apesar de o acesso ser USB 2.0, tem boa veloci..."
80248,comparado ao produto que usava anteriormente é...
61220,Muito bom\n\nO que gostei: Muito instrutivo\n\...
76452,Produto excelente. Sua qualidade de imagem cha...
...,...
6265,acho lindo quero esse modelo de qualquer geito...
54886,todomundo vai querer um ipad como esse
76820,"foi um investimento muito bom, sem arrependime..."
860,"Considero um bom aparelho, com um conceito eco..."


In [16]:
X_train_token['token'] = X_train_token['token'].apply(word_tokenize)

In [17]:
X_train_token['token'] = X_train_token['token'].apply(lambda text: remove_stopwords_punctuation(text))

In [18]:
X_train_token['len_token'] = X_train_token['token'].apply(lambda text: len(text))

In [19]:
X_train_token

,token,len_token
69382,"[Ótimo, aparelho, Para, usa, irá, experimentar...",25
11241,"[Apesar, acesso, USB, 2.0, boa, velocidade, tr...",105
80248,"[comparado, produto, usava, anteriormente, sup...",40
61220,"[Muito, bom, O, gostei, Muito, instrutivo, O, ...",10
76452,"[Produto, excelente, Sua, qualidade, imagem, c...",18
...,...,...
6265,"[acho, lindo, quero, modelo, qualquer, geito, ...",19
54886,"[todomundo, vai, querer, ipad]",4
76820,"[investimento, bom, arrependimento, O, gostei,...",25
860,"[Considero, bom, aparelho, conceito, ecológico...",25


In [20]:
X_train_token = remove_outliers(X_train_token, 'len_token')

In [21]:
X_train_token

,token,len_token
69382,"[Ótimo, aparelho, Para, usa, irá, experimentar...",25
80248,"[comparado, produto, usava, anteriormente, sup...",40
61220,"[Muito, bom, O, gostei, Muito, instrutivo, O, ...",10
76452,"[Produto, excelente, Sua, qualidade, imagem, c...",18
51615,"[MDesign, moderno, todas, tecnologias, necessa...",21
...,...,...
6265,"[acho, lindo, quero, modelo, qualquer, geito, ...",19
54886,"[todomundo, vai, querer, ipad]",4
76820,"[investimento, bom, arrependimento, O, gostei,...",25
860,"[Considero, bom, aparelho, conceito, ecológico...",25


In [22]:
# paridade de índices em X e y
X_train = X_train.loc[X_train_token.index]

In [23]:
y_train = y_train.loc[X_train_token.index]

In [24]:
print(X_train.shape, y_train.shape)

(55151, 7) (55151,)


### 3. Transformação
##### 3.1. Remoção de colunas "inúteis"

In [25]:
X_train.columns

Index(['original_index', 'review_text', 'review_text_processed',
       'review_text_tokenized', 'rating', 'kfold_polarity', 'kfold_rating'],
      dtype='str')

In [26]:
X_train = X_train.drop(columns=['original_index', 'review_text_processed', 'review_text_tokenized', 'rating', 'kfold_polarity', 'kfold_rating'])

In [27]:
print(f'colunas: {X_train.columns}. tipo: {type(X_train)}')

colunas: Index(['review_text'], dtype='str'). tipo: <class 'pandas.DataFrame'>


##### 3.2. Tratamento na label

- Transforma coluna binária em ternária

In [28]:
y_train.value_counts()

polarity
1.0    44023
0.0     3930
Name: count, dtype: int64

In [29]:
y_train.isnull().sum()

np.int64(7198)

In [30]:
y_train = y_train.map({1.0: 2, 0.0: 0})  # {valor_antigo: valor_atual}
y_train = y_train.fillna(1)

In [31]:
y_train.value_counts()

polarity
2.0    44023
1.0     7198
0.0     3930
Name: count, dtype: int64

##### 3.3. Resampling

In [32]:
type(y_train)

pandas.Series

In [33]:
us = RandomUnderSampler(random_state=0)

In [34]:
X_train, y_train = us.fit_resample(X_train, y_train)

In [35]:
print(X_train.shape, y_train.shape, type(X_train), type(y_train))

(11790, 1) (11790,) <class 'pandas.DataFrame'> <class 'pandas.Series'>


In [36]:
y_train.value_counts()

polarity
0.0    3930
1.0    3930
2.0    3930
Name: count, dtype: int64

##### 3.4. PROCESSAMENTO DE LINGUAGEM NATURAL
##### 3.4.1. Tokenização

In [37]:
X_train

,review_text
44756,Comprei o produto faz duas semanas mas só agor...
67481,"Nao compra, som muito ruim e pessima qualidade..."
108,"Péssimo, tem um barulho de cigarra que depois ..."
69342,"Produto inovador com a cara da Apple, acredito..."
80483,Deixa a desejar pelo preço do produto. Possui ...
...,...
56093,Ainda não tive. Mais já experimentei no da min...
41699,Amei o produto. Estou bastante satisfeita\n\nO...
49012,excelente\n\nO que gostei: praticidade em ter ...
17517,"Bom, como adquiri ontem rsrs a minha experiênc..."


In [38]:
X_train['review_text'] = X_train['review_text'].apply(word_tokenize)

In [39]:
X_train

,review_text
44756,"[Comprei, o, produto, faz, duas, semanas, mas,..."
67481,"[Nao, compra, ,, som, muito, ruim, e, pessima,..."
108,"[Péssimo, ,, tem, um, barulho, de, cigarra, qu..."
69342,"[Produto, inovador, com, a, cara, da, Apple, ,..."
80483,"[Deixa, a, desejar, pelo, preço, do, produto, ..."
...,...
56093,"[Ainda, não, tive, ., Mais, já, experimentei, ..."
41699,"[Amei, o, produto, ., Estou, bastante, satisfe..."
49012,"[excelente, O, que, gostei, :, praticidade, em..."
17517,"[Bom, ,, como, adquiri, ontem, rsrs, a, minha,..."


##### 3.4.2. Remoção de stopwords e pontuação

In [40]:
X_train['review_text'] = X_train['review_text'].apply(lambda x: remove_stopwords_punctuation(x))

In [41]:
X_train

,review_text
44756,"[Comprei, produto, faz, duas, semanas, agora, ..."
67481,"[Nao, compra, som, ruim, pessima, qualidade, i..."
108,"[Péssimo, barulho, cigarra, algum, tempom, uso..."
69342,"[Produto, inovador, cara, Apple, acredito, evo..."
80483,"[Deixa, desejar, preço, produto, Possui, opçõe..."
...,...
56093,"[Ainda, Mais, experimentei, amiga, ameei, Já, ..."
41699,"[Amei, produto, Estou, bastante, satisfeita, O..."
49012,"[excelente, O, gostei, praticidade, ter, unica..."
17517,"[Bom, adquiri, ontem, rsrs, experiência, tão, ..."


##### 3.4.3. Lemmatizing

In [42]:
!python -m spacy download pt_core_news_sm

     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
      --------------------------------------- 0.3/13.0 MB ? eta -:--:--
      --------------------------------------- 0.3/13.0 MB ? eta -:--:--
     - ------------------------------------- 0.5/13.0 MB 599.9 kB/s eta 0:00:21
     -- ------------------------------------ 0.8/13.0 MB 621.9 kB/s eta 0:00:20
     -- ------------------------------------ 0.8/13.0 MB 621.9 kB/s eta 0:00:20
     --- ----------------------------------- 1.0/13.0 MB 629.1 kB/s eta 0:00:19
     --- ----------------------------------- 1.0/13.0 MB 629.1 kB/s eta 0:00:19
     --- ----------------------------------- 1.3/13.0 MB 677.8 kB/s eta 0:00:18
     ---- ---------------------------------- 1.6/13.0 MB 705.1 kB/s eta 0:00:17
     ---- ---------------------------------- 1.6/13.0 MB 705.1 kB/s eta 0:00:17



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
nlp = spacy.load("pt_core_news_sm")

In [44]:
X_train["review_text"] = X_train["review_text"].apply(lambda x: lematiza_tokens(x, nlp))

In [45]:
X_train

,review_text
44756,"[Comprei, produto, fazer, dois, semana, agora,..."
67481,"[nao, compra, som, ruim, pessimo, qualidade, i..."
108,"[péssimo, barulho, cigarrar, algum, tempom, us..."
69342,"[produto, inovador, carar, Apple, acreditar, e..."
80483,"[Deixa, desejar, preço, produto, Possui, opção..."
...,...
56093,"[ainda, Mais, experimentar, amigo, ameei, já, ..."
41699,"[Amei, produto, estar, bastante, satisfeito, o..."
49012,"[excelente, o, gostar, praticidade, ter, unico..."
17517,"[bom, adquirir, ontem, rsrs, experiência, tão,..."


##### 3.4.4. Embedding

In [46]:
model_embedding = Word2Vec(X_train['review_text'], min_count=1, vector_size=100, window=5)

In [47]:
X_train['review_text'] = X_train['review_text'].apply(lambda x: embedding_w2v(x, model_embedding))

In [48]:
X_train

,review_text
44756,"[[-0.18705413, 0.21863765, 0.27405658, 0.43017..."
67481,"[[-0.08919807, -0.30372143, 0.5999687, -0.1228..."
108,"[[-0.11274389, 0.22563119, 0.30696592, 0.89389..."
69342,"[[-0.48883912, 0.17755371, 0.035779167, 0.3749..."
80483,"[[-0.050582223, 0.12729855, 0.05202451, 0.1319..."
...,...
56093,"[[-0.49119863, 0.12179836, 0.06475986, 0.31858..."
41699,"[[-0.00931363, 0.0024446836, 0.018159708, 0.04..."
49012,"[[-0.7047386, 0.8070043, 0.43215743, 1.4230739..."
17517,"[[-0.53586906, 0.6329704, 0.48499492, 1.153814..."


### 4. Salvando os dados